# Latches — Cross-Coupled Gates and the Birth of Memory

A latch is the first circuit that *remembers*: two gates feed each other's output back as input, creating a stable loop that holds a bit. This notebook draws the **cross-coupled schematic** with live wire levels and, alongside it, the **timing waveforms** so you see the state being held, set, and reset over time.

$$Q_{next} = f(\text{inputs},\, Q), \qquad \overline{Q} = \text{NOT}(Q)$$


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 9,
})

ON, OFF = '#c0392b', '#b0b0b0'
def wcol(bit): return ON if bit else OFF
def wlw(bit):  return 2.4 if bit else 1.2

def gate_box(ax, x, y, label, w=1.0, h=0.8):
    ax.add_patch(Rectangle((x, y-h/2), w, h, fc='#eef2f7', ec='#34495e', lw=1.4, zorder=2))
    ax.text(x+w/2, y, label, ha='center', va='center', fontsize=8.5, weight='bold', zorder=3)
    return (x, y+h*0.28), (x, y-h*0.28), (x+w, y)

def wire(ax, p0, p1, bit, elbow=True):
    c, lw = wcol(bit), wlw(bit)
    (x0,y0),(x1,y1) = p0, p1
    if elbow and abs(y0-y1) > 1e-6:
        xm = (x0+x1)/2
        ax.plot([x0,xm],[y0,y0], color=c, lw=lw, zorder=1)
        ax.plot([xm,xm],[y0,y1], color=c, lw=lw, zorder=1)
        ax.plot([xm,x1],[y1,y1], color=c, lw=lw, zorder=1)
    else:
        ax.plot([x0,x1],[y0,y1], color=c, lw=lw, zorder=1)

def pin(ax, x, y, name, bit, side='left'):
    ax.scatter([x],[y], s=42, color=wcol(bit), zorder=4)
    dx = -0.3 if side=='left' else 0.2
    ha = 'right' if side=='left' else 'left'
    ax.text(x+dx, y, f'{name}={bit}', ha=ha, va='center', fontsize=9, color=wcol(bit), weight='bold')

def waveform(ax, t, sig, label, color):
    ax.step(t, sig, where='post', color=color, lw=2)
    ax.set_ylim(-0.3, 1.3); ax.set_yticks([0,1])
    ax.set_ylabel(label, rotation=0, ha='right', va='center')
    ax.grid(True, alpha=0.3)

print('primitives ready')


primitives ready


## SR Latch (NOR-Based) — The Cross-Coupled Core

Two NOR gates feed back into each other. $S$ sets $Q=1$, $R$ resets $Q=0$, and with $S=R=0$ the loop **holds** its last value. The combination $S=R=1$ is forbidden: it forces both outputs to 0, breaking $Q=\overline{Q}$.

$$Q = \overline{R + \overline{Q}}, \qquad \overline{Q} = \overline{S + Q}$$


In [ ]:
def sr_nor_settle(S, R, q0):
    """Iterate the cross-coupled loop to its stable state."""
    q, qb = q0, 1 - q0
    for _ in range(10):
        q  = 1 - (R | qb)
        qb = 1 - (S | q)
    forbidden = (S == 1 and R == 1)
    return q, qb, forbidden

def draw_sr_nor(S, R, q0):
    q, qb, forbidden = sr_nor_settle(S, R, q0)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.set_xlim(0, 9); ax.set_ylim(0, 5); ax.axis('off')
    pin(ax, 0.6, 3.6, 'S', S); pin(ax, 0.6, 1.4, 'R', R)
    n1t, n1b, n1o = gate_box(ax, 3.0, 3.4, 'NOR')   # top -> Qbar
    n2t, n2b, n2o = gate_box(ax, 3.0, 1.6, 'NOR')   # bottom -> Q
    wire(ax, (0.6, 3.6), n1t, S)
    wire(ax, (0.6, 1.4), n2b, R)
    # cross-coupling
    wire(ax, n1o, (5.4, 3.4), qb, elbow=False)
    wire(ax, n2o, (5.4, 1.6), q, elbow=False)
    ax.plot([5.4,5.9],[3.4,3.4], color=wcol(qb), lw=wlw(qb))
    ax.plot([5.9,5.9],[3.4,0.9], color=wcol(qb), lw=wlw(qb))
    ax.plot([5.9,2.6],[0.9,0.9], color=wcol(qb), lw=wlw(qb))
    ax.plot([2.6,2.6],[0.9,1.42], color=wcol(qb), lw=wlw(qb))
    ax.plot([5.4,6.2],[1.6,1.6], color=wcol(q), lw=wlw(q))
    ax.plot([6.2,6.2],[1.6,4.1], color=wcol(q), lw=wlw(q))
    ax.plot([6.2,2.6],[4.1,4.1], color=wcol(q), lw=wlw(q))
    ax.plot([2.6,2.6],[4.1,3.68], color=wcol(q), lw=wlw(q))
    pin(ax, 6.2, 3.4, 'Q\u0305', qb, side='right')
    pin(ax, 6.5, 1.6, 'Q', q, side='right')
    title = 'SR Latch (NOR)'
    if forbidden:
        title += '  --  FORBIDDEN S=R=1 (Q = Q\u0305 = 0)'
    ax.set_title(title, fontsize=10, color='#c0392b' if forbidden else 'black')
    plt.tight_layout(); plt.show()

w_S = widgets.ToggleButtons(options=[0,1], value=1, description='S:')
w_R = widgets.ToggleButtons(options=[0,1], value=0, description='R:')
w_q0 = widgets.ToggleButtons(options=[0,1], value=0, description='prev Q:')
display(widgets.VBox([w_S, w_R, w_q0]),
        widgets.interactive_output(draw_sr_nor, {'S': w_S, 'R': w_R, 'q0': w_q0}))


Output()

## SR Latch Timing — Set, Hold, Reset Over Time

Static schematics show one instant; the value of a latch is in its *history*. Here $S$ and $R$ pulse over time and $Q$ tracks them: it goes high on a set pulse, **holds** through the quiet stretches, and drops on a reset pulse.


In [3]:
def sr_timeline(S_pat, R_pat):
    S = np.array([int(c) for c in S_pat.ljust(16,'0')[:16]])
    R = np.array([int(c) for c in R_pat.ljust(16,'0')[:16]])
    t = np.arange(len(S))
    Q = np.zeros(len(S), dtype=int); q = 0
    for i in range(len(S)):
        if S[i] and not R[i]: q = 1
        elif R[i] and not S[i]: q = 0
        elif S[i] and R[i]: q = 0   # forbidden -> both low, show Q=0
        Q[i] = q
    fig, axes = plt.subplots(3, 1, figsize=(8, 3.8), sharex=True)
    waveform(axes[0], t, S, 'S', '#2471a3')
    waveform(axes[1], t, R, 'R', '#2ca02c')
    waveform(axes[2], t, Q, 'Q', '#c0392b')
    # shade forbidden ticks
    for i in range(len(S)):
        if S[i] and R[i]:
            for ax in axes: ax.axvspan(i, i+1, color='#c0392b', alpha=0.12)
    axes[-1].set_xlabel('time tick')
    plt.tight_layout(); plt.show()

w_Spat = widgets.Text(value='0011000000110000', description='S:', layout=widgets.Layout(width='420px'))
w_Rpat = widgets.Text(value='0000001100000011', description='R:', layout=widgets.Layout(width='420px'))
display(widgets.VBox([w_Spat, w_Rpat]),
        widgets.interactive_output(sr_timeline, {'S_pat': w_Spat, 'R_pat': w_Rpat}))


Output()

## Gated D Latch — Transparency Controlled by Enable

Adding an enable gate and tying $S=D$, $R=\overline{D}$ removes the forbidden state. While $EN=1$ the latch is **transparent** ($Q$ follows $D$); while $EN=0$ it **holds**. The schematic lights up the enable path.

$$Q_{next} = \begin{cases} D & EN = 1 \\ Q & EN = 0 \end{cases}$$


In [4]:
def draw_d_latch(D, EN, q0):
    q = D if EN else q0
    qb = 1 - q
    s_int = D & EN          # internal set line
    r_int = (1 - D) & EN    # internal reset line
    fig, ax = plt.subplots(figsize=(7.5, 4))
    ax.set_xlim(0, 9.5); ax.set_ylim(0, 5); ax.axis('off')
    pin(ax, 0.5, 3.7, 'D', D); pin(ax, 0.5, 0.9, 'EN', EN)
    a1t, a1b, a1o = gate_box(ax, 2.3, 3.4, 'AND')   # D & EN
    a2t, a2b, a2o = gate_box(ax, 2.3, 1.6, 'AND')   # ~D & EN
    wire(ax, (0.5, 3.7), a1t, D)
    wire(ax, (0.5, 0.9), a1b, EN)
    wire(ax, (0.5, 0.9), a2b, EN)
    ax.text(1.4, 1.9, 'D\u0305', color=wcol(1-D), fontsize=8, weight='bold')
    wire(ax, (1.6, 1.85), a2t, 1-D)
    n1t, n1b, n1o = gate_box(ax, 5.0, 3.0, 'NOR')
    n2t, n2b, n2o = gate_box(ax, 5.0, 1.4, 'NOR')
    wire(ax, a1o, n1t, s_int)
    wire(ax, a2o, n2b, r_int)
    wire(ax, n2o, (7.2, 1.4), q, elbow=False)
    wire(ax, n1o, (7.2, 3.0), qb, elbow=False)
    pin(ax, 7.2, 3.0, 'Q\u0305', qb, side='right')
    pin(ax, 7.5, 1.4, 'Q', q, side='right')
    mode = 'TRANSPARENT (Q follows D)' if EN else 'HOLD (Q latched)'
    ax.set_title(f'Gated D Latch  --  {mode}', fontsize=10,
                 color='#2471a3' if EN else '#7f8c8d')
    plt.tight_layout(); plt.show()

w_D = widgets.ToggleButtons(options=[0,1], value=1, description='D:')
w_EN = widgets.ToggleButtons(options=[0,1], value=1, description='EN:')
w_dq0 = widgets.ToggleButtons(options=[0,1], value=0, description='prev Q:')
display(widgets.VBox([w_D, w_EN, w_dq0]),
        widgets.interactive_output(draw_d_latch, {'D': w_D, 'EN': w_EN, 'q0': w_dq0}))


Output()

## Transparency in the Waveform — Why a Latch Is Level-Sensitive

This is the plot that distinguishes a *latch* from a *flip-flop*. Wherever $EN=1$ (shaded), $Q$ copies $D$ instantly, including every glitch on $D$. Wherever $EN=0$, $Q$ freezes at the last value. A latch reacts to the **level** of enable, not its edge.


In [5]:
def d_latch_timeline(EN_pat):
    n = 60
    t = np.linspace(0, n, n, endpoint=False).astype(int)
    rng = np.random.default_rng(3)
    D = ((np.sin(t/3.0) > 0).astype(int) ^ (rng.random(n) < 0.06).astype(int))
    # expand EN pattern to length n
    base = [int(c) for c in EN_pat.ljust(10,'0')[:10]]
    EN = np.array([base[i % len(base)] for i in range(n)])
    Q = np.zeros(n, dtype=int); q = 0
    for i in range(n):
        if EN[i]: q = D[i]
        Q[i] = q
    fig, axes = plt.subplots(3, 1, figsize=(8.5, 3.8), sharex=True)
    waveform(axes[0], t, D, 'D', '#2471a3')
    waveform(axes[1], t, EN, 'EN', '#2ca02c')
    waveform(axes[2], t, Q, 'Q', '#c0392b')
    for i in range(n):
        if EN[i]:
            for ax in axes: ax.axvspan(i, i+1, color='#2ca02c', alpha=0.08)
    axes[-1].set_xlabel('time tick   (shaded = transparent window)')
    plt.tight_layout(); plt.show()

w_ENpat = widgets.Text(value='1111000000', description='EN cycle:', layout=widgets.Layout(width='420px'))
display(w_ENpat, widgets.interactive_output(d_latch_timeline, {'EN_pat': w_ENpat}))


Text(value='1111000000', description='EN cycle:', layout=Layout(width='420px'))

Output()

## NOR-Based vs NAND-Based SR — The Duality

The same cross-coupled idea built from NAND gates is the $\overline{S}\,\overline{R}$ latch: it is *active-low* and its forbidden combination is the opposite. The cell tabulates both side by side so the duality is explicit rather than asserted.


In [6]:
def sr_nor(S, R, q0):
    q, qb = q0, 1-q0
    for _ in range(10):
        q  = 1-(R|qb); qb = 1-(S|q)
    return q

def sr_nand(Sb, Rb, q0):
    q, qb = q0, 1-q0
    for _ in range(10):
        q  = 1-(Sb & qb); qb = 1-(Rb & q)
    return q

print('NOR latch (active-high S,R):')
print(f"  {'S':>2}{'R':>3} | action")
for S,R in [(0,0),(0,1),(1,0),(1,1)]:
    act = {('0','0'):'hold',('0','1'):'reset Q=0',('1','0'):'set Q=1',('1','1'):'FORBIDDEN'}[(str(S),str(R))]
    print(f'  {S:>2}{R:>3} | {act}')
print()
print('NAND latch (active-low S\u0305,R\u0305):')
print(f"  {'S\u0305':>2}{'R\u0305':>3} | action")
for Sb,Rb in [(1,1),(1,0),(0,1),(0,0)]:
    act = {(1,1):'hold',(1,0):'reset Q=0',(0,1):'set Q=1',(0,0):'FORBIDDEN'}[(Sb,Rb)]
    print(f'  {Sb:>2}{Rb:>3} | {act}')


NOR latch (active-high S,R):
   S  R | action
   0  0 | hold
   0  1 | reset Q=0
   1  0 | set Q=1
   1  1 | FORBIDDEN

NAND latch (active-low S̅,R̅):
  S̅ R̅ | action
   1  1 | hold
   1  0 | reset Q=0
   0  1 | set Q=1
   0  0 | FORBIDDEN


## Observation: Why Transparency Is Dangerous

| EN | D behaviour | Q behaviour |
|----|-------------|-------------|
| 1  | any change, incl. glitch | passes straight through |
| 0  | any change | ignored, Q held |

If $D$ glitches while $EN=1$, the glitch reaches $Q$ and may be latched when $EN$ falls. The demo injects a single glitch and counts how often it survives, depending on enable timing.


In [7]:
def glitch_survival(glitch_pos, en_fall):
    n = 24
    t = np.arange(n)
    D = np.zeros(n, dtype=int)
    D[glitch_pos] = 1                      # one-tick glitch
    EN = (t < en_fall).astype(int)         # enable high until it falls
    Q = np.zeros(n, dtype=int); q = 0
    for i in range(n):
        if EN[i]: q = D[i]
        Q[i] = q
    latched = Q[-1] == 1
    fig, axes = plt.subplots(3, 1, figsize=(8, 3.6), sharex=True)
    waveform(axes[0], t, D, 'D', '#2471a3')
    waveform(axes[1], t, EN, 'EN', '#2ca02c')
    waveform(axes[2], t, Q, 'Q', '#c0392b')
    axes[0].axvspan(glitch_pos, glitch_pos+1, color='#f39c12', alpha=0.3)
    axes[-1].set_xlabel('time tick')
    msg = 'GLITCH LATCHED -> corrupt state' if latched else 'glitch harmlessly ignored'
    fig.suptitle(msg, color='#c0392b' if latched else '#2ca02c', fontsize=10)
    plt.tight_layout(); plt.show()

w_gp = widgets.IntSlider(value=3, min=0, max=20, description='glitch @:')
w_ef = widgets.IntSlider(value=8, min=1, max=23, description='EN falls @:')
display(widgets.VBox([w_gp, w_ef]),
        widgets.interactive_output(glitch_survival, {'glitch_pos': w_gp, 'en_fall': w_ef}))


Output()